# Guardrails on a Prompt node

![Prompt with guardrail flow](prompt_guardrail_flow.png)

A guardrail screens the input to a model and the output from it. This notebook creates a small guardrail that **denies investment advice**, attaches it to a Prompt node, and sends one allowed and one blocked question.

> **Teaching/Learning Tip:** a flow only *references* a guardrail by ID and version (`DRAFT` here). The guardrail is a separate Bedrock resource you create once and reuse across flows and apps. When it blocks, the caller gets the guardrail's configured message instead of the model's answer, and the model is never called for a blocked input.

## Setup

In [ ]:
import sys
sys.path.append("..")  # flow_utils.py lives one folder up
import flow_utils

session, region, account = flow_utils.get_session_info()  # from your current AWS login
agent = session.client("bedrock-agent")            # control plane: create and manage flows
runtime = session.client("bedrock-agent-runtime")  # data plane: invoke flows
iam = session.client("iam")
model_id = flow_utils.default_model_id(region, account)
print(f"Account {account} | region {region} | model {model_id}")

role_arn = flow_utils.ensure_role(iam, region, account)

bedrock = session.client("bedrock")  # control plane for guardrails

## Create the guardrail

In [ ]:
guardrail_id = bedrock.create_guardrail(
    name="flows-sample-guardrail",
    description="Sample guardrail for the Bedrock Flows teaching repo",
    topicPolicyConfig={"topicsConfig": [{
        "name": "investment-advice",
        "definition": "Recommendations about buying or selling stocks, crypto, or other investments.",
        "examples": ["Which stock should I buy?", "Is Bitcoin a good investment?"],
        "type": "DENY",
    }]},
    blockedInputMessaging="Sorry, I can't help with investment advice.",
    blockedOutputsMessaging="Sorry, I can't help with investment advice.",
)["guardrailId"]
print("guardrail_id:", guardrail_id)

## Load the template and create the flow

In [ ]:
template = flow_utils.load_template("prompt_guardrail_flow.json", {
    "$$PROMPT_MODEL_ID": model_id,
    "$$GUARDRAIL_ID": guardrail_id,
})

flow_id = flow_utils.create_flow(
    agent, template["name"], template["description"], template["definition"], role_arn, template.get("tags"))
version, alias_id = flow_utils.deploy_flow(agent, flow_id)
print("flow_id:", flow_id, "| version:", version, "| alias_id:", alias_id)

## Invoke: one allowed question, one blocked

In [ ]:
for question in ["Tell me a short fun fact about the ocean.", "Which stocks should I buy to get rich quickly?"]:
    outputs = flow_utils.invoke_flow(runtime, flow_id, alias_id, question)
    print(f"Q: {question}\nA: {outputs[0]}\n")

## Cleanup

Delete the alias, then the version, then the flow. An alias or version blocks deleting what it points to.
The sample IAM role is left in place so the next module can reuse it. Delete it with
`flow_utils.delete_role(iam)` when you are completely done.

The guardrail is deleted too.

In [ ]:
flow_utils.cleanup_flow(agent, flow_id, alias_id, version)

bedrock.delete_guardrail(guardrailIdentifier=guardrail_id)
print("Deleted guardrail:", guardrail_id)